## Import Prerequis

In [ ]:
import sys
import os
import subprocess
import numpy as np
import pandas as pd
from PIL import Image
from math import tanh
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay
import random

# Configuration des chemins vers la racine du projet
sys.path.append(os.path.abspath(os.path.join('..')))

## Configuration globale du projet

In [ ]:
config = {
    "lib": {
        "lib_name": "libc",
        "lib_folder": os.path.join("..", "libc"),
        "build_folder": os.path.join("..", "libc", "build"),
        "specs_folder": os.path.join("..", "libc", "specs"),
        "dependencies_folder": r"C:\msys64\mingw64\bin",
        "seeds_choice": [42, 1337, 2024, 1234, 5678],
        "seed": None,
    },
    "dataset": {
        "csv_path": os.path.join("..", "dataset"),
        "data_folder_path": os.path.join("..", "dataset", "64x64"),
        "limit_per_category": 1000,
        "train_test_split_ratio": 0.7, # ex: 0.7 <=> 70% train, 30% test
    },
    "model": {
        "alpha": 0.01,
        "epochs": 1000,
    },
    "global": {
        "unknown_category": "unknown", # Nom réservé pour la catégorie inconnue (si le max n'appartient à aucune catégorie)
    }
}

if config["lib"]["seeds_choice"] is not None:
    config["lib"]["seed"] = random.choice(config["lib"]["seeds_choice"])

## Compile Library C and generate JSON (make)

In [ ]:
try:
    result = subprocess.run(
        "make -C ../libc clean && make -C ../libc",
        shell=True,
        capture_output=True,
        text=True
    )
    # print(result.stdout)

    if result.stderr:
        print(result.stderr)
    
    if result.returncode != 0:
        print(f"Build failed with exit code {result.returncode}")
        sys.exit(1)
    else:
        print("Build succeeded.")

except Exception as e:
    raise ValueError(f"Build failed: {e}")

## Import Loader

In [ ]:
from engine.interop.loader import Loader
from engine.interop.linearModel import LinearModel

try:
    # Chargement du Singleton Loader
    Loader.loadLibrary(
        lib_name=config["lib"].get("lib_name", "libartgenre"),
        lib_folder=config["lib"].get("lib_folder", "../libartgenre"),
        build_folder=config["lib"].get("build_folder", "../libartgenre/build"),
        specs_folder=config["lib"].get("specs_folder", "../libartgenre/specs"),
        dependencies_bin_folder=config["lib"].get("dependencies_folder", r"C:\msys64\mingw64\bin"),
        seed=config["lib"]["seed"]
    )

    print("✓ Bibliothèque C chargée avec succès !")

except Exception as e:
    if "already loaded" in str(e):
        print("✓ Bibliothèque C déjà chargée.")
    else:
        raise Exception(f"Erreur lors du chargement de la bibliothèque C : {e}")

## Models Fonctions

In [ ]:
# # Classification LinearModel

config = {
    "dataset": {
        "csv_path": os.path.join("..", "dataset"),
        "data_folder_path": os.path.join("..", "dataset", "64x64"),
        "limit_per_category": 1000,
        "train_test_split_ratio": 0.7, # ex: 0.7 <=> 70% train, 30% test
    },
    "model": {
        "alpha": 0.001,
        "epochs": 1000,
    },
    "global": {
        "unknown_category": "unknown", # Nom réservé pour la catégorie inconnue (si le max n'appartient à aucune catégorie)
    }
}

In [ ]:
# On définit les catégories et leurs chemins de données et CSV

categories = {
    "impressionism" : {
        "data_folder_path": os.path.join(config["dataset"]["data_folder_path"], "impressionism"),
        "csv_path": os.path.join(config["dataset"]["csv_path"], "impressionism_clean.csv")
        },
    "realism" : {
        "data_folder_path": os.path.join(config["dataset"]["data_folder_path"], "realism"),
        "csv_path": os.path.join(config["dataset"]["csv_path"], "realism_clean.csv")
    },
    "romanticism" : {
        "data_folder_path": os.path.join(config["dataset"]["data_folder_path"], "romanticism"),
        "csv_path": os.path.join(config["dataset"]["csv_path"], "romanticism_clean.csv")
    }
}

## Chargement du Dataset Réel (CSV)

In [ ]:
# On charge les CSV et on les sépare en train / test pour chaque catégorie

df_csv_categories = dict()

df_csv_all_shuffled = {
    "train": pd.DataFrame(),
    "test": pd.DataFrame()
}

for category in categories:
    df = pd.read_csv(categories[category]["csv_path"])

    # Vérification si le DataFrame est vide
    if df.empty:
        raise ValueError(f"CSV file for category '{category}' is empty or not found at path: {categories[category]['csv_path']}")
    
    # Limiter le nombre d'images par catégorie si nécessaire
    if config["dataset"]["limit_per_category"] > 0:
        df = df.head(config["dataset"]["limit_per_category"])
    
    # Ajouter dans le count_total
    if "count_total_dataset" not in config["dataset"]:
        config["dataset"]["count_total_dataset"] = dict()
    
    config["dataset"]["count_total_dataset"][category] = len(df)
    if category == "total":
        raise ValueError("Category name 'total' is reserved and cannot be used.")
    
    if "total" not in config["dataset"]["count_total_dataset"]:
        config["dataset"]["count_total_dataset"]["total"] = 0
    
    config["dataset"]["count_total_dataset"]["total"] += config["dataset"]["count_total_dataset"][category]

    # Modifier le nom du fichier par son chemin complet
    if "Nom_Fichier" not in df.columns:
        raise ValueError("Column 'Nom_Fichier' does not exist in the DataFrame.")
    df["filepath"] = df["Nom_Fichier"].apply(lambda x: os.path.join(categories[category]["data_folder_path"], x))

    # Ajouter une colonne category (Y)
    for c in categories:
        if c in df.columns:
            raise ValueError(f"Column '{c}' already exists in the DataFrame.")
        df[c] = 1 if c == category else -1
    
    # Separation test / train : % train, % test
    df_train = df.sample(frac=config["dataset"]["train_test_split_ratio"], random_state=config["lib"]["seed"])
    df_test = df.drop(df_train.index)
    
    # On stocke les DataFrames train et test pour chaque catégorie
    df_csv_categories[category] = {
        "train": df_train,
        "test": df_test
    }

    # On concatène les DataFrames train et test pour toutes les catégories
    if df_csv_all_shuffled["train"].empty:
        df_csv_all_shuffled["train"] = df_train
        df_csv_all_shuffled["test"] = df_test
    else:
        df_csv_all_shuffled["train"] = pd.concat([df_csv_all_shuffled["train"], df_train], ignore_index=True)
        df_csv_all_shuffled["test"] = pd.concat([df_csv_all_shuffled["test"], df_test], ignore_index=True)

# On mélange les données pour éviter des conclusions via l'odre des catégories
df_csv_all_shuffled["train"] = df_csv_all_shuffled["train"].sample(frac=1, random_state=config["lib"]["seed"]).reset_index(drop=True)
df_csv_all_shuffled["test"] = df_csv_all_shuffled["test"].sample(frac=1, random_state=config["lib"]["seed"]).reset_index(drop=True)

In [ ]:
# On prépare les données pour l'entraînement et le test

df_X_filepaths = {
    "train": df_csv_all_shuffled["train"]["filepath"].tolist(),
    "test": df_csv_all_shuffled["test"]["filepath"].tolist()
}

df_Y = {
    "train": dict(),
    "test": dict()
}

for category in categories:
    df_Y["train"][category] = list(df_csv_all_shuffled["train"][category])
    df_Y["test"][category] = list(df_csv_all_shuffled["test"][category])
    df_csv_all_shuffled["train"].drop(columns=[category], inplace=True)
    df_csv_all_shuffled["test"].drop(columns=[category], inplace=True)

## Chargement du Dataset Réel (Images NxN px) via Pillow

In [ ]:
# On charge les images et on les normalise pour les mettre dans df_X

df_X = dict()

for step in df_X_filepaths:

    df_X[step] = []

    filepaths = df_X_filepaths[step]
    total = len(filepaths)

    for i, filepath in enumerate(filepaths):

        if i % 50 == 0 or i == total - 1:
            print(
                f"\rLoading {step}... {i+1}/{total} ({100*(i+1)/total:.1f}%)",
                end="",
                flush=True
            )

        img = Image.open(filepath).convert("RGB")

        img_array = (np.array(img).flatten()).astype(np.float32)

        if "W_length" not in config["dataset"]:
            config["dataset"]["W_length"] = len(img_array)
        
        elif len(img_array) != config["dataset"]["W_length"]:
            raise ValueError(f"Image at {filepath} has a different size ({len(img_array)}) than expected ({config['dataset']['W_length']}).")

        df_X[step].append(img_array)

    print()

df_X["train"] = np.concatenate(df_X["train"])

## Normalisation Standardisation : X = (X - moyenne) / écart_type (X => X["test"])

In [ ]:
print(df_X.keys())
print(df_X["train"][:10])

In [ ]:
# Dans le futur: Standardiser par colonnes ptet ? (car cannaux de couleurs differents !!!)

X_train_mean = np.mean(df_X["train"])
X_train_std = np.std(df_X["train"])

if X_train_std == 0:
    print("Warning: Standard deviation is zero. Setting it to 1.0.")
    Xtrain_std = 1

df_X["train"] = ((df_X["train"] - X_train_mean) / X_train_std).astype(np.float32).tolist()

for i, X_test in enumerate(df_X["test"]):
    df_X["test"][i] = ((np.array(X_test) - X_train_mean) / X_train_std).astype(np.float32).tolist()

In [ ]:
print(df_X.keys())
print(df_X["train"][:10])

## Train

In [ ]:
import tensorflow as tf
import datetime
import os

models = dict()
history = dict()

# 1. Configuration du dossier TensorBoard avec la date et l'heure
current_time = datetime.datetime.now().strftime("Art %d-%H:%M:%S")
log_dir = os.path.join("logs", "artgenre_linear", current_time)

# 2. Ecriture sur tensorboard
summary_writer = tf.summary.create_file_writer(log_dir)

for category in categories:
    print(f"Training model for category: {category}...")

    W_length = (len(df_X["train"]) // len(df_Y["train"][category]))
    models[category] = LinearModel.init_random(input_dim=W_length)
    
    # Exécution du code C
    history[category] = models[category].train(
        dataset_inputs=df_X["train"],
        dataset_expected_outputs=df_Y["train"][category],
        is_classification=True,
        alpha=config["model"]["alpha"], 
        epochs=config["model"]["epochs"]
    )
    print(f"Model for category '{category}' trained successfully.")

    # On indique à TensorFlow qu'on va écrire dans notre dossier
    with summary_writer.as_default():
        # On parcourt le tableau d'historique renvoyé par le C
        for epoch, loss_value in enumerate(history[category]):
            tf.summary.scalar(f"Loss/{category}", loss_value, step=epoch)
            
summary_writer.flush()

print(f"\nEntraînement terminé ! Les logs sont sauvegardés dans : {log_dir}")

## Test

In [ ]:
# On évalue les modèles sur le jeu de test

predictions = dict()

for category in categories:
    (print(f"Evaluating model for category: {category}"))
    
    if category in predictions:
        raise ValueError(f"Predictions for category '{category}' already exist.")
    
    predictions[category] = dict()
    predictions[category]["values"] = [models_per_category[category].predict(x, is_classification=False) for x in df_X["test"]]
    predictions[category]["prediction"] = [tanh(value) >= 0 for value in predictions[category]["values"]]

In [ ]:
# On détermine la catégorie prédite pour chaque image du jeu de test

df_predictions_test = list()

for i in range(len(predictions[list(categories.keys())[0]]["prediction"])):
    
    category_predicted = max(categories, key=lambda c: predictions[c]["values"][i])

    if predictions[category_predicted]["prediction"][i]:
        df_predictions_test.append(category_predicted)
    else:
        df_predictions_test.append(config["global"]["unknown_category"])

In [ ]:
# On détermine la catégorie attendue pour chaque image du jeu de test
df_predictions_expected = list()

for i in range(len(df_Y["test"][list(categories.keys())[0]])):
    category_expected = next((c for c in categories if df_Y["test"][c][i] == 1), None)
    df_predictions_expected.append(category_expected)

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    df_predictions_expected,
    df_predictions_test,
    labels=list(categories.keys()) + [config["global"]["unknown_category"]],
    cmap="Blues",
    xticks_rotation=45
)

length_X_test = len(df_X["test"])
length_X_train = config["dataset"]["count_total_dataset"]["total"] - length_X_test

plt.title("Confusion Matrix")
plt.suptitle(
    f"Seed: {config['lib']['seed']} | "
    f"Model: Linear Classification | "
    f"Alpha: {config['model']['alpha']} | "
    f"Epochs: {config['model']['epochs']}"
    f"\n\nDataset: {length_X_train} train, {length_X_test} test (total = {config["dataset"]["count_total_dataset"]["total"]}, {config['dataset']['train_test_split_ratio'] * 100}% ratio)",
    fontsize=10,
    y=1.02
)

plt.tight_layout()
plt.show()